## Analysis Pipeline
This notebook loads data, runs the analysis loop, computes derived statistics, and saves all results.

In [1]:
import matplotlib
matplotlib.use('Agg')
#Imports
from glob import glob
import numpy as np
import pandas as pd
import os
from matplotlib import pyplot as plt
import cv2
from tqdm import tqdm_notebook as tqdm
import random
import ast

%run loom_analysis_functions.py

#Paths
machine = 'hilbert'

if machine == 'adams':
    PROJECT_PATH = '/ssd02/Projects/Data/jen_projects/virtualcricket_loom/'
elif machine == 'hilbert':
    PROJECT_PATH = '/media/arnab/My Book Duo/Data/hoylab_projects/virtualcricket_loom_claude_cleaned/'

## Analysis Parameters
Change these values to adjust the analysis. All downstream cells and the plots notebook will use them.

In [2]:
max_seconds_from_first_loom = 120  # seconds from first loom to include (set to None for all looms)
speeds = [1, 6, 25]
pad_onset = 30
pad_offset = 60
fps = 30
loom_duration = 15 * 3
escape_thresholds = {'distance': 100, 'speed': 25, 'duration': 200}

In [3]:
looming_videos_mp4 = glob(os.path.join(PROJECT_PATH,'*','*', '*.mp4'))
looming_videos_avi = glob(os.path.join(PROJECT_PATH,'*','*', '*.avi'))
looming_videos = looming_videos_mp4 + looming_videos_avi
dlc_labels = glob(os.path.join(PROJECT_PATH,'DLC_out/*/*', '*.h5'))
metadata_df = pd.read_csv(os.path.join(PROJECT_PATH, 'virtualcricket_loom_metadata.csv'))

arena_labels = glob(os.path.join(PROJECT_PATH, 'combined_arena_shelter_looming_coordinates.csv'))
arena_labels_df = pd.read_csv(arena_labels[0])

In [4]:
if False:
    # Generate and save loom labels
    looming_timestamps = {}
    for video in tqdm(looming_videos):
        row = arena_labels_df[arena_labels_df['video_file'] == os.path.basename(video)]
        loom_center = np.array(row[['looming_point1_x', 'looming_point1_y', 'looming_point2_x', 'looming_point2_y', 'looming_point3_x', 'looming_point3_y', 'looming_point4_x', 'looming_point4_y']])
        loom_center = loom_center.reshape(-1, 2)
        output = detect_luminance_change(video, loom_center, threshold_percent=9, min_area_percent=60)
        filename = os.path.basename(video).split('.')[0]
        print(filename)
        print(row.video_file)
        # Combine outputs that are close together
        combined_output = []
        if len(output) > 0:
            current_group = [output[0]]
            for i in range(1, len(output)):
                if output[i][0] - current_group[-1][0] < 90:
                    current_group.append(output[i])
                else:
                    combined_output.append((current_group[0][0], current_group[-1][1]))
                    current_group = [output[i]]
            combined_output.append((current_group[0][0], current_group[-1][1]))
        looming_timestamps[filename] =combined_output

    # Save the looming timestamps dictionary as npz
    np.savez(os.path.join(PROJECT_PATH, 'looming_timestamps.npz'), **looming_timestamps)

In [5]:
# Load the npz file
looming_timestamps = dict(np.load(os.path.join(PROJECT_PATH, 'looming_timestamps.npz'), allow_pickle=True))
len(looming_timestamps)

286

## Metadata Parsing & Group Assignment

In [6]:
mice_id ={}

In [7]:
#Let's parse the metadata file here
#We have four main groups of mice: for initial analysis - Males, Females, Adults and Adolescents

mice_id['males'] = set(metadata_df[metadata_df['sex'] == 'M']['id'])
mice_id['females'] = set(metadata_df[metadata_df['sex'] == 'F']['id'])
mice_id['adults'] = set(metadata_df[metadata_df['age'] == 'P60']['id'])
mice_id['adolescents'] = set(metadata_df[metadata_df['age'] == 'P45']['id'])
mice_id['sheltered'] = set(metadata_df[metadata_df['shelter_used'] == 'Y']['id'])
mice_id['unsheltered'] = set(metadata_df[metadata_df['shelter_used'] == 'N']['id'])
mice_id['experienced'] = set(metadata_df[metadata_df['naive'] == 'N']['id'])
mice_id['naive'] = set(metadata_df[metadata_df['naive'] == 'Y']['id'])


In [8]:
# take the intersection of the sets

mice_id['experienced_adult_males_shelter'] = list(mice_id['adults'].intersection(mice_id['males']).intersection(mice_id['sheltered']).intersection(mice_id['experienced']))
mice_id['experienced_adult_females_shelter'] = list(mice_id['adults'].intersection(mice_id['females']).intersection(mice_id['sheltered']).intersection(mice_id['experienced']))
mice_id['experienced_adolescent_males_shelter'] = list(mice_id['adolescents'].intersection(mice_id['males']).intersection(mice_id['sheltered']).intersection(mice_id['experienced']))
mice_id['experienced_adolescent_females_shelter'] = list(mice_id['adolescents'].intersection(mice_id['females']).intersection(mice_id['sheltered']).intersection(mice_id['experienced']))

mice_id['experienced_adult_males_unsheltered'] = list(mice_id['adults'].intersection(mice_id['males']).intersection(mice_id['unsheltered']).intersection(mice_id['experienced']))
mice_id['experienced_adult_females_unsheltered'] = list(mice_id['adults'].intersection(mice_id['females']).intersection(mice_id['unsheltered']).intersection(mice_id['experienced']))
mice_id['experienced_adolescent_males_unsheltered'] = list(mice_id['adolescents'].intersection(mice_id['males']).intersection(mice_id['unsheltered']).intersection(mice_id['experienced']))
mice_id['experienced_adolescent_females_unsheltered'] = list(mice_id['adolescents'].intersection(mice_id['females']).intersection(mice_id['unsheltered']).intersection(mice_id['experienced']))

#naive mice
mice_id['naive_adult_males_shelter'] = list(mice_id['naive'].intersection(mice_id['adults']).intersection(mice_id['males']).intersection(mice_id['sheltered']))
mice_id['naive_adult_females_shelter'] = list(mice_id['naive'].intersection(mice_id['adults']).intersection(mice_id['females']).intersection(mice_id['sheltered']))
mice_id['naive_adolescent_males_shelter'] = list(mice_id['naive'].intersection(mice_id['adolescents']).intersection(mice_id['males']).intersection(mice_id['sheltered']))
mice_id['naive_adolescent_females_shelter'] = list(mice_id['naive'].intersection(mice_id['adolescents']).intersection(mice_id['females']).intersection(mice_id['sheltered']))

In [9]:
print(f'{len(mice_id["experienced_adult_males_shelter"])} experienced adult males in shelter')
print(f'{len(mice_id["experienced_adult_females_shelter"])} experienced adult females in shelter')
print(f'{len(mice_id["experienced_adolescent_males_shelter"])} experienced adolescent males in shelter')
print(f'{len(mice_id["experienced_adolescent_females_shelter"])} experienced adolescent females in shelter')

print('--------------------------------')

print(f'{len(mice_id["experienced_adult_males_unsheltered"])} experienced adult males without shelter')
print(f'{len(mice_id["experienced_adult_females_unsheltered"])} experienced adult females without shelter')
print(f'{len(mice_id["experienced_adolescent_males_unsheltered"])} experienced adolescent males without shelter')
print(f'{len(mice_id["experienced_adolescent_females_unsheltered"])} experienced adolescent females without shelter')

print('--------------------------------')

print(f'{len(mice_id["naive_adult_males_shelter"])} naive adult males in shelter')
print(f'{len(mice_id["naive_adult_females_shelter"])} naive adult females in shelter')
print(f'{len(mice_id["naive_adolescent_males_shelter"])} naive adolescent males in shelter')
print(f'{len(mice_id["naive_adolescent_females_shelter"])} naive adolescent females in shelter')

8 experienced adult males in shelter
8 experienced adult females in shelter
8 experienced adolescent males in shelter
8 experienced adolescent females in shelter
--------------------------------
2 experienced adult males without shelter
2 experienced adult females without shelter
2 experienced adolescent males without shelter
2 experienced adolescent females without shelter
--------------------------------
10 naive adult males in shelter
10 naive adult females in shelter
9 naive adolescent males in shelter
9 naive adolescent females in shelter


In [10]:
# VALIDATION: Check if sex and groups are assigned correctly
# This validates that each mouse in each group actually has the correct metadata attributes

print("=" * 80)
print("VALIDATION: Checking sex and group assignments")
print("=" * 80)

# Define the expected attributes for each group
group_attributes = {
    'experienced_adult_males_shelter': {'sex': 'M', 'age': 'P60', 'naive': 'N', 'shelter_used': 'Y'},
    'experienced_adult_females_shelter': {'sex': 'F', 'age': 'P60', 'naive': 'N', 'shelter_used': 'Y'},
    'experienced_adolescent_males_shelter': {'sex': 'M', 'age': 'P45', 'naive': 'N', 'shelter_used': 'Y'},
    'experienced_adolescent_females_shelter': {'sex': 'F', 'age': 'P45', 'naive': 'N', 'shelter_used': 'Y'},
    'experienced_adult_males_unsheltered': {'sex': 'M', 'age': 'P60', 'naive': 'N', 'shelter_used': 'N'},
    'experienced_adult_females_unsheltered': {'sex': 'F', 'age': 'P60', 'naive': 'N', 'shelter_used': 'N'},
    'experienced_adolescent_males_unsheltered': {'sex': 'M', 'age': 'P45', 'naive': 'N', 'shelter_used': 'N'},
    'experienced_adolescent_females_unsheltered': {'sex': 'F', 'age': 'P45', 'naive': 'N', 'shelter_used': 'N'},
    'naive_adult_males_shelter': {'sex': 'M', 'age': 'P60', 'naive': 'Y', 'shelter_used': 'Y'},
    'naive_adult_females_shelter': {'sex': 'F', 'age': 'P60', 'naive': 'Y', 'shelter_used': 'Y'},
    'naive_adolescent_males_shelter': {'sex': 'M', 'age': 'P45', 'naive': 'Y', 'shelter_used': 'Y'},
    'naive_adolescent_females_shelter': {'sex': 'F', 'age': 'P45', 'naive': 'Y', 'shelter_used': 'Y'}
}

errors_found = []
warnings_found = []

# Check each group
for group_name, expected_attrs in group_attributes.items():
    if group_name not in mice_id:
        warnings_found.append(f"WARNING: Group '{group_name}' not found in mice_id dictionary")
        continue
    
    mouse_list = mice_id[group_name]
    
    for mouse_id_val in mouse_list:
        # Get the actual metadata for this mouse
        mouse_metadata = metadata_df[metadata_df['id'] == mouse_id_val]
        
        if len(mouse_metadata) == 0:
            errors_found.append(f"ERROR: Mouse '{mouse_id_val}' in group '{group_name}' not found in metadata!")
            continue
        
        if len(mouse_metadata) > 1:
            warnings_found.append(f"WARNING: Mouse '{mouse_id_val}' appears multiple times in metadata")
        
        mouse_row = mouse_metadata.iloc[0]
        
        # Check each attribute
        for attr_name, expected_value in expected_attrs.items():
            actual_value = mouse_row[attr_name]
            if str(actual_value) != str(expected_value):
                errors_found.append(
                    f"ERROR: Mouse '{mouse_id_val}' in group '{group_name}' has {attr_name}='{actual_value}' "
                    f"but expected '{expected_value}'"
                )

# Check for mice that should be in groups but aren't
print("\nChecking for mice that should be in groups but are missing...")
all_grouped_mice = set()
for group_name in group_attributes.keys():
    if group_name in mice_id:
        all_grouped_mice.update(mice_id[group_name])

all_metadata_mice = set(metadata_df['id'].unique())
ungrouped_mice = all_metadata_mice - all_grouped_mice

if ungrouped_mice:
    print(f"\nWARNING: {len(ungrouped_mice)} mice in metadata are not assigned to any group:")
    for mouse_id_val in sorted(ungrouped_mice):
        mouse_metadata = metadata_df[metadata_df['id'] == mouse_id_val].iloc[0]
        print(f"  - {mouse_id_val}: sex={mouse_metadata['sex']}, age={mouse_metadata['age']}, "
              f"naive={mouse_metadata['naive']}, shelter_used={mouse_metadata['shelter_used']}")

# Print summary
print("\n" + "=" * 80)
if errors_found:
    print(f" !!!!!!! VALIDATION FAILED: Found {len(errors_found)} errors:")
    for error in errors_found[:20]:  # Show first 20 errors
        print(f"  {error}")
    if len(errors_found) > 20:
        print(f"  ... and {len(errors_found) - 20} more errors")
else:
    print("✅ VALIDATION PASSED: All mice are correctly assigned to groups based on metadata")
    
if warnings_found:
    print(f"\n !!!!!WARNINGS: Found {len(warnings_found)} warnings:")
    for warning in warnings_found[:10]:  # Show first 10 warnings
        print(f"  {warning}")
    if len(warnings_found) > 10:
        print(f"  ... and {len(warnings_found) - 10} more warnings")

print("=" * 80)

# Also check the basis for assignment
print("\nBASIS FOR ASSIGNMENT:")
print("  - Sex: From metadata_df['sex'] column (M = male, F = female)")
print("  - Age: From metadata_df['age'] column (P60 = adult, P45 = adolescent)")
print("  - Experience: From metadata_df['naive'] column (Y = naive, N = experienced)")
print("  - Shelter: From metadata_df['shelter_used'] column (Y = sheltered, N = unsheltered)")
print("  - Groups: Created by taking intersections of the above sets")
print("=" * 80)

VALIDATION: Checking sex and group assignments

Checking for mice that should be in groups but are missing...

✅ VALIDATION PASSED: All mice are correctly assigned to groups based on metadata

BASIS FOR ASSIGNMENT:
  - Sex: From metadata_df['sex'] column (M = male, F = female)
  - Age: From metadata_df['age'] column (P60 = adult, P45 = adolescent)
  - Experience: From metadata_df['naive'] column (Y = naive, N = experienced)
  - Shelter: From metadata_df['shelter_used'] column (Y = sheltered, N = unsheltered)
  - Groups: Created by taking intersections of the above sets


In [11]:
%run loom_analysis_functions.py

## Main Analysis Loop

In [12]:
if True:
    results = {}
    blender_data = {}
    rejected_mice = []

    # only analyzing with shelter data for now
    group_names = ['naive_adult_males_shelter', 'naive_adult_females_shelter', 'naive_adolescent_males_shelter',
                    'naive_adolescent_females_shelter', 'experienced_adult_males_shelter',
                    'experienced_adult_females_shelter', 'experienced_adolescent_males_shelter',
                    'experienced_adolescent_females_shelter']

    false_positives = 0
    for group_name in group_names:
        
        if group_name not in results:
            results[group_name] = {}
        else:
            print(f'{group_name} already in results')
        
        for mouse_id in mice_id[group_name]:
            speed_order = get_speed_for_id(metadata_df, mouse_id)

            # Global time budget: single clock across all speed sessions
            if max_seconds_from_first_loom is not None:
                remaining_budget_frames = max_seconds_from_first_loom * fps
            else:
                remaining_budget_frames = None
            first_loom_found = False
            first_loom_frame_offset = None

            for speed in speed_order:
                video_pattern = f'/{mouse_id}_{speed}.'
                matching_videos = [v for v in looming_videos if video_pattern in v]
                if not matching_videos or len(matching_videos) > 1:
                    print(f"Warning: No video found or multiple videos found for {mouse_id}_{speed}")
                    continue

                looming_video = matching_videos[0]
                video_filename = looming_video.split('.')[0].split('/')[-1]

                #need to pull frame timing here   
                dlc_label = match_videos_to_labels(looming_video, dlc_labels)
                arena_kpts, shelter_kpts, loom_center_kpts, row = get_arena_shelter_loom_labels(looming_video, arena_labels)
                loom_center_location = np.mean(loom_center_kpts, axis=0) #WIP: make this more robust
                shelter_location = np.mean(shelter_kpts, axis=0)

                #Compute scale factor (pixels to cm)
                pix2cm_scale = get_pix2cm_scale(arena_kpts)
                
                #get looming timestamps
                if video_filename not in looming_timestamps:
                    print(f"Warning: No looming timestamps found for {video_filename}, skipping...")
                    continue
                loom_times = looming_timestamps[video_filename]

                # Apply global time budget filter
                if remaining_budget_frames is not None:
                    if not first_loom_found:
                        if len(loom_times) > 0:
                            first_loom_frame_offset = loom_times[0][0]
                            first_loom_found = True
                            loom_times = [lt for lt in loom_times if (lt[0] - first_loom_frame_offset) <= remaining_budget_frames]
                        else:
                            loom_times = []
                    else:
                        if remaining_budget_frames <= 0:
                            loom_times = []
                        else:
                            loom_times = [lt for lt in loom_times if lt[0] <= remaining_budget_frames]

                dlc_df = pd.read_hdf(dlc_label)
                scorer = dlc_df.keys()[0][0]
                dlc_df = dlc_df[scorer].__deepcopy__()

                n_looms = len(loom_times)
            
                #Continuous variables trialwise
                mouse_centpos_trialwise = [] #mouse centroid positions for each trial
                cricket_pos_trialwise = [] #cricket positions for each trial
                mouse_cricket_distance_trialwise = [] # distance of mouse from cricket for each trial
                mouse_shelter_distance_trialwise = [] # distance of mouse from shelter for each trial
                mouse_speed_trialwise = [] # speed of mouse for each trial
                heading_mouse_trialwise = [] # heading of mouse for each trial in radians
                heading_cricket_mouse_trialwise = [] # heading of cricket relative to mouse for each trial azimuth in radians
                loom_azimuth_trialwise = [] # position of looming stimulus azimuth for each trial in radians
                loom_elevation_trialwise = [] # position of looming stimulus elevation for each trial in radians
                cricket_speed_trialwise = [] # speed of cricket for each trial
                #discrete variables
                escape_to_shelter_trialwise = [] # whether mouse escaped to shelter
                freeze_trialwise = [] # whether mouse froze
                
                #if mouse escapes to shelter then compute
                escape_latency_trialwise = [] # escape latency
                max_escape_speed_trialwise = [] # maximum speed of escape
                mean_escape_speed_trialwise = [] # mean speed of escape
                #if mouse freezes then compute
                freeze_duration_trialwise = [] # duration of freeze
                trial_info = []  #0- False Positive, 1- Escape, 2- Freeze, 3- No Escape or Freeze
                if n_looms != 0:
                    blender_data[video_filename] = {}
                    blender_data[video_filename]['group'] = group_name
                    blender_data[video_filename]['stimulus'] = []
                    blender_data[video_filename]['midear_x'] = []
                    blender_data[video_filename]['midear_y'] = []
                    blender_data[video_filename]['heading'] = []
                    blender_data[video_filename]['escape_to_shelter'] = []
                    blender_data[video_filename]['trial_info'] = []
                    blender_data[video_filename]['speed'] = speed

                    corrected_dlc_df = correct_stimulus_coordinates(dlc_df, arena_kpts, pix2cm_scale = pix2cm_scale)
                    heading_df = correct_heading_coordinates(dlc_df, likelihood_threshold = 0.4)
                    # Counts legit looms discounting false positives
                    #Loop through each trial (which is every time time a looming stimulus is presented)
                    for j, loom_time in enumerate(loom_times):
                        loom_start, loom_end = loom_time
                        #loom_duration = loom_end - loom_start # not using this for now as this is inacurate.
                        #  Using loom onset + some duration
                        loom_end = loom_start + loom_duration
                        trial_start, trial_end, pad_onset_actual, pad_offset_actual, trial_duration = correct_trial_timestamps(loom_start,
                                                                                                            loom_end,
                                                                                                            pad_onset,
                                                                                                            pad_offset,
                                                                                                            dlc_df)

                        mouse_centpos_trial, cricket_pos_trial = get_mouse_and_cricket_positions(dlc_df, trial_start, trial_end)

                        # #Check for false positives
                        true_positive_result = check_loom_true_positive(dlc_df,
                                    mouse_centpos_trial,
                                    cricket_pos_trial,
                                    loom_start = pad_onset_actual,
                                    pix2cm_scale=pix2cm_scale,
                                    approach_distance_threshold = 16, 
                                    approach_binocular_threshold = 40, #30
                                    min_tracking_frames = 0, #3
                                    min_tracking_percentage = 0.1, #0.5
                                    verbose = True)
                        if isinstance(true_positive_result, tuple):
                            true_positive = true_positive_result[0]
                        else:
                            true_positive = true_positive_result

                        if not true_positive:
                            blender_data[video_filename]['trial_info'].append(0)
                            false_positives += 1 #we should still take into account that a loom was presented - say for habituation
                            trial_info.append(0)
                            continue

                        #position data for blender WIP: Redundant stuff here need to remove
                        blender_data[video_filename]['stimulus'].append(corrected_dlc_df.loc[trial_start:trial_end, ('stimulus_distance')].values)
                        blender_data[video_filename]['midear_x'].append(interpolate_nans(np.nanmean(np.vstack((corrected_dlc_df.loc[trial_start:trial_end, ('leftear', 'x')].values, corrected_dlc_df.loc[trial_start:trial_end, ('rightear', 'x')].values)), axis=0)))
                        blender_data[video_filename]['midear_y'].append(interpolate_nans(np.nanmean(np.vstack((corrected_dlc_df.loc[trial_start:trial_end, ('leftear', 'y')].values, corrected_dlc_df.loc[trial_start:trial_end, ('rightear', 'y')].values)), axis=0)))
                        #need to deal with heading nan
                        blender_heading = get_mouse_heading(heading_df, (trial_start, trial_end), return_cricket_azimuth = False)
                        blender_heading = [-value for value in blender_heading]
                        blender_data[video_filename]['heading'].append(blender_heading)

                        #Compute the continuous variables
                        mouse_centpos_trialwise.append(mouse_centpos_trial)
                        cricket_pos_trialwise.append(cricket_pos_trial)
                        mouse_cricket_distance_trialwise.append(mouse_dynamic_object_distance(mouse_centpos_trial, cricket_pos_trial))

                        #Subtract distance equal to max dist between shelter mean and arena kpts
                        mouse_shelter_distance_trial = mouse_static_object_distance(mouse_centpos_trial, shelter_location)
                        mouse_shelter_distance_trialwise.append(mouse_shelter_distance_trial)
                        
                        # WIP Use a better way to get the speed, add timestamps for each frame
                        mouse_speed_trial = mouse_speed(mouse_centpos_trial)*fps/pix2cm_scale # in cm/s
                        mouse_speed_trialwise.append(mouse_speed_trial) 
                        cricket_speed_trialwise.append(speed)
                        #Compute the heading and stim positions
                        headings = get_mouse_heading(dlc_df, (trial_start, trial_end), return_cricket_azimuth = True)   
                        mouse_heading, cricket_azimuth = zip(*headings)                    
                        heading_cricket_mouse = ((np.array(cricket_azimuth) - np.array(mouse_heading) + np.pi) % (2*np.pi)) - np.pi #recheck this formula
                        heading_mouse_trialwise.append(np.array(mouse_heading))
                        heading_cricket_mouse_trialwise.append(np.array(heading_cricket_mouse))
                        
                        #this function below may need an overhaul
                        loom_azimuth = []
                        loom_elevation = []
                        for frame_idx in range(trial_start, trial_end):
                            tmp_loom_azimuth, tmp_loom_elevation = get_mouse_loom_position(dlc_df,frame_idx,
                                                                                loom_center_location,
                                                                                pix2cm_scale = pix2cm_scale)
                            tmp_mouse_heading = get_mouse_heading(dlc_df, frame_idx, return_cricket_azimuth = False)
                            tmp_loom_pos = ((np.array(tmp_mouse_heading) - np.array(tmp_loom_azimuth) + np.pi) % (2*np.pi)) - np.pi #recheck this formula

                            loom_azimuth.append(tmp_loom_pos)
                            loom_elevation.append(tmp_loom_elevation)
                        loom_azimuth_trialwise.append(loom_azimuth)
                        loom_elevation_trialwise.append(loom_elevation)

                        #Compute the categorical variables
                        escape_to_shelter_trialwise.append(escape_to_shelter(mouse_shelter_distance_trial[pad_onset_actual:],
                                                                            mouse_speed_trial[pad_onset_actual:],
                                                                            verbose = True, 
                                                                            escape_distance_threshold = escape_thresholds['distance'],
                                                                            escape_speed_threshold = escape_thresholds['speed'],
                                                                            escape_duration_threshold = escape_thresholds['duration']))

                        if escape_to_shelter_trialwise[-1]:
                            trial_info.append(1)
                            blender_data[video_filename]['escape_to_shelter'].append(1)
                            blender_data[video_filename]['trial_info'].append(1)
                            escape_dict = analyze_mouse_escape(distance = mouse_shelter_distance_trial,
                                                            speed = mouse_speed_trial,
                                                            fps = fps,  # frames per second
                                                            freeze_speed_threshold = 10,  # cm/s
                                                            escape_speed_threshold = 10.0,  # cm/s
                                                            shelter_distance_threshold = escape_thresholds['distance'],
                                                            min_freeze_duration_frames = 3,  # minimum frames for a valid freeze
                                                            min_escape_duration_frames = 3)  # minimum frames for a valid escape)

                            freeze_duration = escape_dict['freeze_end_frame'] - escape_dict['freeze_start_frame']
                            # Use .get() with default values in case escape periods weren't detected
                            escape_duration = escape_dict.get('escape_end_frame', 0) - escape_dict.get('escape_start_frame', 0)
                            escape_avg_speed = escape_dict.get('escape_avg_speed', np.nan)
                            escape_max_speed = escape_dict.get('escape_max_speed', np.nan)

                            escape_latency_trialwise.append(escape_dict.get('escape_start_frame', 0))
                            # compute max and mean escape speed
                            max_escape_speed_trialwise.append(escape_max_speed)
                            mean_escape_speed_trialwise.append(escape_avg_speed)
                            freeze_trialwise.append(False)
                            freeze_duration_trialwise.append(np.nan)
                        else:
                            blender_data[video_filename]['escape_to_shelter'].append(0)
                            freeze, duration = detect_freeze(mouse_shelter_distance_trial[pad_onset_actual:],
                                                            mouse_speed_trial[pad_onset_actual:],
                                                            fps = fps,
                                                            freeze_speed_threshold = 2.5,
                                                            min_freeze_duration_frames = 15)
                            freeze_trialwise.append(freeze)
                            freeze_duration_trialwise.append(duration)
                            escape_latency_trialwise.append(np.nan)
                            max_escape_speed_trialwise.append(np.nan)
                            mean_escape_speed_trialwise.append(np.nan)
                            if freeze:
                                trial_info.append(2)
                                blender_data[video_filename]['trial_info'].append(2)
                            else:
                                trial_info.append(3)
                                blender_data[video_filename]['trial_info'].append(3)
                        
                if video_filename not in results[group_name]:
                    results[group_name][video_filename] = {}
                else:
                    print(f'WARNING: {video_filename} already in results')

                results[group_name][video_filename]['mouse_cricket_distance_trialwise'] = mouse_cricket_distance_trialwise
                results[group_name][video_filename]['mouse_shelter_distance_trialwise'] = mouse_shelter_distance_trialwise
                results[group_name][video_filename]['mouse_speed_trialwise'] = mouse_speed_trialwise
                results[group_name][video_filename]['heading_mouse_trialwise'] = heading_mouse_trialwise
                results[group_name][video_filename]['heading_cricket_mouse_trialwise'] = heading_cricket_mouse_trialwise
                results[group_name][video_filename]['escape_to_shelter_trialwise'] = escape_to_shelter_trialwise
                results[group_name][video_filename]['freeze_trialwise'] = freeze_trialwise
                results[group_name][video_filename]['freeze_duration_trialwise'] = freeze_duration_trialwise
                results[group_name][video_filename]['max_speed_trialwise'] = max_escape_speed_trialwise
                results[group_name][video_filename]['mean_speed_trialwise'] = mean_escape_speed_trialwise
                results[group_name][video_filename]['mouse_centpos_trialwise'] = mouse_centpos_trialwise
                results[group_name][video_filename]['cricket_pos_trialwise'] = cricket_pos_trialwise
                results[group_name][video_filename]['cricket_speed_trialwise'] = cricket_speed_trialwise
                results[group_name][video_filename]['escape_latency_trialwise'] = escape_latency_trialwise
                results[group_name][video_filename]['loom_azimuth_trialwise'] = loom_azimuth_trialwise
                results[group_name][video_filename]['loom_elevation_trialwise'] = loom_elevation_trialwise
                results[group_name][video_filename]['arena_kpts'] = arena_kpts
                results[group_name][video_filename]['shelter_kpts'] = shelter_kpts
                results[group_name][video_filename]['loom_center'] = loom_center_location
                results[group_name][video_filename]['pix2cm_scale'] = pix2cm_scale
                results[group_name][video_filename]['loom_times'] = loom_times
                results[group_name][video_filename]['video_path'] = looming_video.split(PROJECT_PATH)[-1]
                results[group_name][video_filename]['trial_info'] = trial_info

                # Update global time budget after this session
                if remaining_budget_frames is not None and first_loom_found:
                    session_total_frames = len(dlc_df)
                    if first_loom_frame_offset is not None:
                        remaining_budget_frames -= (session_total_frames - first_loom_frame_offset)
                        first_loom_frame_offset = None
                    else:
                        remaining_budget_frames -= session_total_frames

            # Rejection check: exclude mice with zero true-positive looms in first two speeds
            first_two_speeds = speed_order[:2] if len(speed_order) >= 2 else speed_order
            total_true_positives = 0
            for spd in first_two_speeds:
                vf = f'{mouse_id}_{spd}'
                if vf in results[group_name]:
                    ti = results[group_name][vf].get('trial_info', [])
                    total_true_positives += sum(1 for t in ti if t != 0)
            if total_true_positives == 0:
                reason = f"Zero true-positive looms in first two speeds ({first_two_speeds})"
                rejected_mice.append({'mouse_id': mouse_id, 'group': group_name, 'reason': reason,
                                      'first_two_speeds': str(first_two_speeds)})
                for spd in speeds:
                    vf = f'{mouse_id}_{spd}'
                    if vf in results[group_name]:
                        del results[group_name][vf]
                    if vf in blender_data:
                        del blender_data[vf]

    # Remove rejected mice from mice_id so downstream cells skip them
    for entry in rejected_mice:
        mid = entry['mouse_id']
        grp = entry['group']
        if mid in mice_id[grp]:
            mice_id[grp].remove(mid)

    # Print rejection table
    if rejected_mice:
        print(f"\n{'='*80}")
        print(f"REJECTED MICE: {len(rejected_mice)} mice excluded")
        print(f"{'='*80}")
        print(f"{'Mouse ID':<15} {'Group':<45} {'Reason'}")
        print(f"{'-'*15} {'-'*45} {'-'*40}")
        for entry in rejected_mice:
            print(f"{entry['mouse_id']:<15} {entry['group']:<45} {entry['reason']}")
        print(f"{'='*80}")
        
        # Save rejection table as CSV
        import pandas as pd
        rejected_df = pd.DataFrame(rejected_mice)
        rejected_df.to_csv('rejected_mice.csv', index=False)
        print(f"Rejection table saved to rejected_mice.csv")
    else:
        print("\nNo mice rejected - all had true-positive looms in first two speeds.")

No frames found where mouse is within 100 of shelter.
Minimum mouse distance was: 280.6914869556332
No frames found where mouse is within 100 of shelter.
Minimum mouse distance was: 314.198348063569
False positive: Tracking criteria not met
Maximum consecutive tracking frames: 1 (minimum required: 0)
Tracking percentage: 2.50% (minimum required: 10.00%)
False positive: Tracking criteria not met
Maximum consecutive tracking frames: 1 (minimum required: 0)
Tracking percentage: 2.50% (minimum required: 10.00%)
False positive: Mouse was not within 16 cm of cricket during loom window
Minimum distance observed: 16.19 cm
No frames found where mouse is within 100 of shelter.
Minimum mouse distance was: 307.00687989358084
No frames found where mouse is within 100 of shelter.
Minimum mouse distance was: 281.0286410547431
No frames found where mouse is within 100 of shelter.
Minimum mouse distance was: 395.6708116063382
No frames found where mouse is within 100 of shelter.
Minimum mouse distance 

## Per-Mouse Statistics

In [13]:
# only analyzing with shelter data for now
group_names = ['naive_adult_males_shelter', 'naive_adult_females_shelter', 'naive_adolescent_males_shelter',
                'naive_adolescent_females_shelter', 'experienced_adult_males_shelter',
                'experienced_adult_females_shelter', 'experienced_adolescent_males_shelter',
                'experienced_adolescent_females_shelter']

In [14]:
mouse_stats = {}
for group_name in group_names:
    mouse_stats[group_name] = {}
    for mouse_id in mice_id[group_name]:
        #get speed order
        speed_order = get_speed_for_id(metadata_df, mouse_id)
        distance_travelled = []
        time_spent_in_shelter = []
        start_and_end_time = []
        number_of_approaches = []
        escape_array = []
        freeze_array = []
        for speed in speed_order:
            video_pattern = f'/{mouse_id}_{speed}.'
            matching_videos = [v for v in looming_videos if video_pattern in v]
            if not matching_videos or len(matching_videos) > 1:
                print(f"Warning: No video found or multiple videos found for {mouse_id}_{speed}")
                continue

            looming_video = matching_videos[0]
            video_filename = looming_video.split('.')[0].split('/')[-1]
            dlc_label = match_videos_to_labels(looming_video, dlc_labels)

            dlc_df = pd.read_hdf(dlc_label)
            scorer = dlc_df.keys()[0][0]
            dlc_df = dlc_df[scorer].__deepcopy__()
            arena_kpts, shelter_kpts, loom_center_kpts, row = get_arena_shelter_loom_labels(looming_video, arena_labels)
            shelter_location = np.mean(shelter_kpts, axis=0)
            pix2cm_scale = get_pix2cm_scale(arena_kpts)

            distance_travelled.append(get_distance_travelled(dlc_df, pix2cm_scale))
            time_spent_in_shelter.append(get_time_in_shelter(dlc_df, shelter_location))
            start_and_end_time.append(get_vid_start_and_end_time(dlc_df))
            number_of_approaches.append(len(results[group_name][f'{mouse_id}_{speed}']['trial_info']))
            trial_info = np.array(results[group_name][f'{mouse_id}_{speed}']['trial_info'])
            valid_trials = trial_info != 0
            freeze_trials = np.where((trial_info == 2) & valid_trials, 1, 0)[valid_trials]
            escape_trials = np.where((trial_info == 1) & valid_trials, 1, 0)[valid_trials]
            freeze_array.extend(freeze_trials.tolist())
            escape_array.extend(escape_trials.tolist())


        mouse_stats[group_name][mouse_id] = {'distance_travelled': distance_travelled,
                                            'time_spent_in_shelter': time_spent_in_shelter,
                                            'start_and_end_time': start_and_end_time,
                                            'number_of_approaches': number_of_approaches,
                                            'freeze_array': freeze_array,
                                            'escape_array': escape_array}

In [15]:
mouse_stats_aggregated = {}
for group_name in group_names:
    distance_travelled = []
    time_spent_in_shelter = []
    number_of_approaches = []
    freeze_array = []
    escape_array = []
    mouse_stats_aggregated[group_name] = {}
    for mouse_id in mouse_stats[group_name].keys():
        distance_travelled.append(sum(mouse_stats[group_name][mouse_id]['distance_travelled']))
        time_spent_in_shelter.append(sum([time for time, _ in mouse_stats[group_name][mouse_id]['time_spent_in_shelter']]))
        number_of_approaches.append(sum(mouse_stats[group_name][mouse_id]['number_of_approaches']))
        freeze_array.append(mouse_stats[group_name][mouse_id]['freeze_array'])
        escape_array.append(mouse_stats[group_name][mouse_id]['escape_array'])
    mouse_stats_aggregated[group_name]['distance_travelled'] = distance_travelled
    mouse_stats_aggregated[group_name]['time_spent_in_shelter'] = time_spent_in_shelter
    mouse_stats_aggregated[group_name]['number_of_approaches'] = number_of_approaches
    mouse_stats_aggregated[group_name]['freeze_array'] = freeze_array
    mouse_stats_aggregated[group_name]['escape_array'] = escape_array


## Ethogram Data

In [16]:
mouse_ethogram = {}
for group_name in group_names:
    mouse_ethogram[group_name] = {}
    for mouse_id in mice_id[group_name]:
        #get speed order
        speed_order = get_speed_for_id(metadata_df, mouse_id)
        freeze_times = []
        escape_times = []
        no_response_times = []
        false_alarm_times = []
        for i, speed in enumerate(speed_order):
            start_time = mouse_stats[group_name][mouse_id]['start_and_end_time'][i]['start_frame']
            end_time = mouse_stats[group_name][mouse_id]['start_and_end_time'][i]['end_frame']

            vf = f'{mouse_id}_{speed}'
            if vf not in results[group_name]:
                continue
            loom_times = results[group_name][vf]['loom_times']
            if len(loom_times) == 0:
                continue
            corrected_loom_times = [t[0] - start_time for t in loom_times]
            total_samples = end_time - start_time

            resampled_loom_times = resample_events_to_9000(corrected_loom_times, total_samples) + i*9000
            trial_info = results[group_name][vf]['trial_info']
            freeze_times.extend(resampled_loom_times[np.array(trial_info)==2])
            escape_times.extend(resampled_loom_times[np.array(trial_info)==1])
            no_response_times.extend(resampled_loom_times[np.array(trial_info)==3])
            false_alarm_times.extend(resampled_loom_times[np.array(trial_info)==0])
        mouse_ethogram[group_name][mouse_id] = {'freeze_times': freeze_times,
                                                'escape_times': escape_times,
                                                'no_response_times': no_response_times,
                                                'false_alarm_times': false_alarm_times}

In [17]:
ethogram_data = {}
for group_name in group_names:
    escape_times_ethogram = []
    no_response_times_ethogram = []
    freeze_times_ethogram = []
    false_alarm_times_ethogram = []
    for mouse_id in mouse_ethogram[group_name].keys():
        escape_times_ethogram.append(mouse_ethogram[group_name][mouse_id]['escape_times'])
        no_response_times_ethogram.append(mouse_ethogram[group_name][mouse_id]['no_response_times'])
        freeze_times_ethogram.append(mouse_ethogram[group_name][mouse_id]['freeze_times'])
        false_alarm_times_ethogram.append(mouse_ethogram[group_name][mouse_id]['false_alarm_times'])
    ethogram_data[group_name] = {'escape_times': escape_times_ethogram,
                                 'no_response_times': no_response_times_ethogram,
                                 'freeze_times': freeze_times_ethogram,
                                 'false_alarm_times': false_alarm_times_ethogram}

## Export to CSV

In [18]:
import pandas as pd
import numpy as np

# Create an empty list to store the data rows
data_rows = []

# Iterate through the results structure
for group_name in results.keys():
    for video_filename in results[group_name].keys():
        # Extract mouse ID from the video filename
        mouse_id = video_filename.split('_')[0]
        
        # Extract litter from mouse_id (part before the hyphen)
        litter = mouse_id.split('-')[0]

        #extract speed from mouse_id (part after the underscore)
        video_speed = video_filename.split('_')[1]
        
        # Get sex, age, and experience from metadata_df (more reliable than parsing group_name)
        # This ensures we use the actual metadata values rather than inferring from group_name
        mouse_metadata = metadata_df[metadata_df['id'] == mouse_id]
        
        if len(mouse_metadata) == 0:
            print(f"WARNING: Mouse '{mouse_id}' not found in metadata, using group_name inference")
            # Fallback to group_name parsing if metadata not found
            if 'females' in group_name:
                sex = 'female'
            else:
                sex = 'male'
            if 'adolescent' in group_name:
                age = 'adolescent'
            else:
                age = 'adult'
            if 'experienced' in group_name:
                experience = 'experienced'
            else:
                experience = 'naive'
        else:
            # Use actual metadata values
            mouse_row = mouse_metadata.iloc[0]
            
            # Map metadata values to output format
            sex_map = {'M': 'male', 'F': 'female'}
            sex = sex_map.get(mouse_row['sex'], 'unknown')
            
            age_map = {'P60': 'adult', 'P45': 'adolescent'}
            age = age_map.get(mouse_row['age'], 'unknown')
            
            experience_map = {'N': 'experienced', 'Y': 'naive'}
            experience = experience_map.get(mouse_row['naive'], 'unknown')
            
            # Validate against group_name (should match, but warn if it doesn't)
            expected_sex = 'female' if 'females' in group_name else 'male'
            expected_age = 'adolescent' if 'adolescent' in group_name else 'adult'
            expected_experience = 'experienced' if 'experienced' in group_name else 'naive'
            
            if sex != expected_sex or age != expected_age or experience != expected_experience:
                print(f"WARNING: Mouse '{mouse_id}' in group '{group_name}' has metadata mismatch:")
                print(f"  Group suggests: sex={expected_sex}, age={expected_age}, experience={expected_experience}")
                print(f"  Metadata shows: sex={sex}, age={age}, experience={experience}")

        # Get trial data
        latencies_list = results[group_name][video_filename]['escape_latency_trialwise']
        freeze_list = results[group_name][video_filename]['freeze_trialwise']
        freeze_duration_list = results[group_name][video_filename]['freeze_duration_trialwise']
        escape_list = results[group_name][video_filename]['escape_to_shelter_trialwise']
        speeds_max_list = results[group_name][video_filename]['max_speed_trialwise']
        loom_az_list = results[group_name][video_filename]['loom_azimuth_trialwise']
        loom_el_list = results[group_name][video_filename]['loom_elevation_trialwise']
        cricket_az_list = results[group_name][video_filename]['heading_cricket_mouse_trialwise']
        
        # Create a row for each trial
        for i in range(len(escape_list)):
            # Only proceed if we have data for this trial #THIS LOOKS ODD - SKIPS ALL DATA IF ANY ONE OF THESE IS MISSING
            if i >= len(latencies_list) or i >= len(speeds_max_list) or i >= len(loom_az_list) or i >= len(loom_el_list) or i >= len(cricket_az_list):
                print(f"WARNING: Trial {i} for {video_filename} has mismatched list lengths - skipping. "
                      f"escape={len(escape_list)}, latency={len(latencies_list)}, speeds={len(speeds_max_list)}, "
                      f"loom_az={len(loom_az_list)}, loom_el={len(loom_el_list)}, cricket_az={len(cricket_az_list)}")
                continue
            else:
                # Create a data row
                row = {
                    'mouse_id': mouse_id,
                    'litter': litter,
                    'video_speed': video_speed,
                    'sex': sex,
                    'age': age,
                    'experience': experience,
                    'escape': escape_list[i],  # Include the escape boolean
                    'freeze': freeze_list[i],
                    'freeze_duration': freeze_duration_list[i] if i < len(freeze_duration_list) else np.nan,
                    'latency': latencies_list[i] if i < len(latencies_list) else np.nan,
                    'escape_spds_max': speeds_max_list[i] if i < len(speeds_max_list) else np.nan,
                    'loom_az': loom_az_list[i][0] if i < len(loom_az_list) else np.nan,
                    'loom_el': loom_el_list[i][0] if i < len(loom_el_list) else np.nan,
                    'cricket_az': cricket_az_list[i][0] if i < len(cricket_az_list) else np.nan
                }
                data_rows.append(row)

# Create a DataFrame from the rows
df = pd.DataFrame(data_rows)

# Display the first few rows to check
print(df.head())

# Save to CSV file for use in R
df.to_csv('mouse_escape_data_alldata_final-22December.csv', index=False)

# Optional: Save to an R-friendly format
try:
    # If pyreadr is installed
    import pyreadr
    pyreadr.write_rds('mouse_escape_data.rds', df)
    print("RDS file saved for R")
except ImportError:
    print("For an R-friendly RDS format, install pyreadr with: pip install pyreadr")
    print("CSV file saved which can be read in R with: read.csv('mouse_escape_data.csv')")

  mouse_id litter video_speed   sex    age experience  escape  freeze  \
0   R53-36    R53           1  male  adult      naive   False    True   
1   R53-36    R53           1  male  adult      naive   False    True   
2   W46-33    W46           1  male  adult      naive   False    True   
3   W46-33    W46           1  male  adult      naive   False    True   
4    R56-9    R56          25  male  adult      naive   False    True   

   freeze_duration  latency  escape_spds_max   loom_az   loom_el  cricket_az  
0         0.866667      NaN              NaN -1.819976  1.273084    0.569104  
1         1.733333      NaN              NaN  2.309304  1.371171    0.237748  
2         1.666667      NaN              NaN -0.820506  1.404974   -0.738512  
3         1.700000      NaN              NaN -1.847011  1.286346    0.159453  
4         0.733333      NaN              NaN -0.216976  1.406011   -0.193911  
RDS file saved for R


## Save All Results

In [19]:
# Create separate escape arrays for each group as a dict
escape_arrays_by_group = {}
freeze_arrays_by_group = {}

for group_name in group_names:
    group_escape_arrays = mouse_stats_aggregated[group_name]['escape_array']
    group_freeze_arrays = mouse_stats_aggregated[group_name]['freeze_array']
    
    # Determine number of columns based on group name
    num_columns = 10 if 'experienced' in group_name else 5
    
    # Pad arrays for this group to the determined number of columns
    escape_array_padded_group = []
    freeze_array_padded_group = []
    for arr in group_escape_arrays:
        padded_arr = arr + [np.nan] * (num_columns - len(arr))
        # Trim to the determined number of columns if longer
        padded_arr = padded_arr[:num_columns]
        escape_array_padded_group.append(padded_arr)
    for arr in group_freeze_arrays:
        padded_arr = arr + [np.nan] * (num_columns - len(arr))
        # Trim to the determined number of columns if longer
        padded_arr = padded_arr[:num_columns]
        freeze_array_padded_group.append(padded_arr)
    
    # Convert to numpy array
    escape_array_np = np.array(escape_array_padded_group)
    escape_arrays_by_group[group_name] = escape_array_np
    freeze_array_np = np.array(freeze_array_padded_group)
    freeze_arrays_by_group[group_name] = freeze_array_np

escape_arrays_by_group

{'naive_adult_males_shelter': array([[ 0.,  0., nan, nan, nan],
        [ 0.,  0., nan, nan, nan],
        [ 0.,  0.,  0.,  0., nan],
        [ 0.,  1., nan, nan, nan],
        [ 1.,  0., nan, nan, nan],
        [ 0., nan, nan, nan, nan],
        [ 1.,  0., nan, nan, nan],
        [ 0., nan, nan, nan, nan],
        [ 0., nan, nan, nan, nan]]),
 'naive_adult_females_shelter': array([[ 0., nan, nan, nan, nan],
        [ 0.,  1., nan, nan, nan],
        [ 1., nan, nan, nan, nan],
        [ 0., nan, nan, nan, nan],
        [ 1.,  1., nan, nan, nan],
        [ 0., nan, nan, nan, nan],
        [ 0.,  0., nan, nan, nan],
        [ 0., nan, nan, nan, nan],
        [ 0., nan, nan, nan, nan],
        [ 0.,  0., nan, nan, nan]]),
 'naive_adolescent_males_shelter': array([[ 0., nan, nan, nan, nan],
        [ 1., nan, nan, nan, nan],
        [ 0.,  0., nan, nan, nan],
        [ 1., nan, nan, nan, nan],
        [ 1.,  0., nan, nan, nan],
        [ 1.,  0., nan, nan, nan],
        [ 1.,  1., nan, nan

In [20]:
import os

save_dir = os.path.join(PROJECT_PATH, 'analysis_results')
os.makedirs(save_dir, exist_ok=True)

# Save results dict
np.savez(os.path.join(save_dir, 'results.npz'), **{'results': results}, allow_pickle=True)

# Save blender data
np.savez(os.path.join(save_dir, 'blender_data.npz'), **{'blender_data': blender_data}, allow_pickle=True)

# Save mouse stats
np.savez(os.path.join(save_dir, 'mouse_stats.npz'), **{'mouse_stats': mouse_stats}, allow_pickle=True)

# Save mouse stats aggregated
np.savez(os.path.join(save_dir, 'mouse_stats_aggregated.npz'), **{'mouse_stats_aggregated': mouse_stats_aggregated}, allow_pickle=True)

# Save ethogram data
np.savez(os.path.join(save_dir, 'ethogram_data.npz'), **{'ethogram_data': ethogram_data}, allow_pickle=True)

# Save escape arrays
np.savez(os.path.join(save_dir, 'escape_arrays.npz'), **{'escape_arrays_by_group': escape_arrays_by_group, 'freeze_arrays_by_group': freeze_arrays_by_group}, allow_pickle=True)

# Save group_names, mice_id, and analysis parameters for the plotting notebook
analysis_params = {
    'max_seconds_from_first_loom': max_seconds_from_first_loom,
    'fps': fps,
    'pad_onset': pad_onset,
    'pad_offset': pad_offset,
    'loom_duration': loom_duration,
    'escape_thresholds': escape_thresholds,
    'speeds': speeds
}
np.savez(os.path.join(save_dir, 'metadata.npz'),
         **{'group_names': group_names, 'mice_id': mice_id, 'analysis_params': analysis_params},
         allow_pickle=True)

print(f"All results saved to {save_dir}")
for f in os.listdir(save_dir):
    fpath = os.path.join(save_dir, f)
    size_mb = os.path.getsize(fpath) / 1024 / 1024
    print(f"  {f}: {size_mb:.1f} MB")

All results saved to /media/arnab/My Book Duo/Data/hoylab_projects/virtualcricket_loom/analysis_results
  results.npz: 2.6 MB
  blender_data.npz: 1.0 MB
  mouse_stats.npz: 1.8 MB
  mouse_stats_aggregated.npz: 0.0 MB
  ethogram_data.npz: 0.0 MB
  escape_arrays.npz: 0.0 MB
  metadata.npz: 0.0 MB
